# ALife — Experiment: Comparing 3 environments

**Goal**: Measure how the type of environment affects the evolutionary dynamics of a population of artificial agents.

**Starting hypothesis**: A heterogeneous environment (Perlin) should slow down convergence toward slow agents compared to a uniform one (Flat), because fast agents have a local advantage near oases. Drought, by making resources migrate, should create a non-monotonic selection pressure on speed.

---

**Shared parameters**: `seed=42`, `N_init=15`, `ticks=2500`, `regen=0.05`, `mutation_std=0.06`

**Tested modes**: `flat` | `perlin` | `drought`

## 0. Setup

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor'  : '#161b22',
    'axes.edgecolor'  : '#30363d',
    'axes.labelcolor' : '#8b949e',
    'xtick.color'     : '#8b949e',
    'ytick.color'     : '#8b949e',
    'text.color'      : '#c9d1d9',
    'grid.color'      : '#30363d',
    'grid.linewidth'  : 0.5,
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'font.family'     : 'monospace',
})

COLORS = {
    'flat'   : '#58a6ff',   # blue — control
    'perlin' : '#3fb950',   # green — static terrain
    'drought': '#f78166',   # orange — dynamic terrain
}

# Project root (where this notebook lives)
# Experiments generated with:
#   python run.py --mode flat    --out-dir experiments/exp_flat
#   python run.py --mode perlin  --out-dir experiments/exp_perlin
#   python run.py --mode drought --out-dir experiments/exp_drought
BASE = Path('.')
EXPS = {
    'flat'   : BASE / 'experiments/exp_flat',
    'perlin' : BASE / 'experiments/exp_perlin',
    'drought': BASE / 'experiments/exp_drought',
}

def load(exp_key):
    """Load results.json + timeseries.json for a mode."""
    r   = json.loads((EXPS[exp_key] / 'results.json').read_text())
    ts  = json.loads((EXPS[exp_key] / 'timeseries.json').read_text())
    return r, ts

data = {k: load(k) for k in EXPS}
print('Data loaded:')
for k, (r, _) in data.items():
    print(f'  {k:8s}  pop_final={r["pop_final"]:4d}  '
          f'spd_drift={r["speed_drift"]:+.3f}  '
          f'lineages={r["lineages_final"]}/{r["lineages_initial"]}')

## 1. Summary table

A quick look at the raw numbers before visualizing. This table is what we'll comment on in the conclusion.

In [ ]:
from IPython.display import HTML

rows = ''
metrics = [
    ('Final pop.',         lambda r: r['pop_final']),
    ('Max pop.',           lambda r: r['pop_max']),
    ('Stable pop. (mean)', lambda r: f"{r['pop_stable_mean']:.1f}"),
    ('Initial speed',      lambda r: f"{r['speed_initial']:.3f}"),
    ('Final speed',        lambda r: f"{r['speed_final']:.3f}"),
    ('Speed drift',        lambda r: f"{r['speed_drift']:+.3f}"),
    ('Speed diversity σ',  lambda r: f"{r['speed_std_final']:.3f}"),
    ('Surviving lineages', lambda r: f"{r['lineages_final']}/{r['lineages_initial']}"),
    ('Max generation',     lambda r: r['max_generation']),
    ('Mean res. (end)',     lambda r: f"{r['res_mean_final']:.3f}"),
    ('Extinction',         lambda r: r['extinction'] or 'no'),
]

header = '<tr style="background:#21262d"><th>Metric</th>' + ''.join(
    f'<th style="color:{COLORS[k]}">{k.upper()}</th>' for k in EXPS
) + '</tr>'

for label, fn in metrics:
    vals = [str(fn(data[k][0])) for k in EXPS]
    rows += f'<tr><td>{label}</td>' + ''.join(f'<td>{v}</td>' for v in vals) + '</tr>'

style = 'style="border-collapse:collapse;width:100%;font-family:monospace;font-size:13px"'
td_style = 'style="padding:6px 12px;border-bottom:1px solid #30363d"'
HTML(f'<table {style}><thead>{header}</thead><tbody>'
     + rows.replace('<td>', f'<td {td_style}>') + '</tbody></table>')

## 2. Population dynamics

**What to look for**: growth speed and the equilibrium plateau. A higher plateau means the environment supports more individuals (carrying capacity). Faster early growth means less survival pressure at the start.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

for mode in EXPS:
    r, ts_data = data[mode]
    pop = ts_data['timeseries']['pop']
    T   = range(len(pop))
    ax.plot(T, pop, color=COLORS[mode], lw=1.8, label=mode, alpha=0.9)
    ax.fill_between(T, pop, alpha=0.07, color=COLORS[mode])
    ax.annotate(f" {pop[-1]}", xy=(len(pop)-1, pop[-1]),
                color=COLORS[mode], fontsize=9, va='center')

ax.axhline(15, color='#8b949e', lw=0.7, ls='--', alpha=0.5)
ax.text(2, 16, 'N₀=15', color='#8b949e', fontsize=8)
ax.set_xlabel('Tick'); ax.set_ylabel('Population')
ax.set_title('Population by environment', pad=10)
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

## 3. Genetic evolution — mean speed

**This is the central graph.** Speed is the only heritable genetic trait. Its trajectory reveals everything about selection pressure.

- **Fast drop** → the environment penalizes fast agents (too costly)
- **Slow drop** → fast agents have a compensatory advantage (reaching oases)
- **Plateau or rebound** → equilibrium or pressure reversal (drought)

The transparent envelope = min/max range = genetic diversity within the population.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, mode in zip(axes, EXPS):
    r, ts_data = data[mode]
    ts  = ts_data['timeseries']
    T   = range(len(ts['speed_mean']))
    spd = ts['speed_mean']
    smin, smax = ts['speed_min'], ts['speed_max']

    ax.plot(T, spd, color=COLORS[mode], lw=2, label='mean')
    ax.fill_between(T, spd, alpha=0.18, color=COLORS[mode])
    ax.fill_between(T, smin, smax, alpha=0.07, color=COLORS[mode],
                    label='min–max range')

    ax.axhline(spd[0],  color='#8b949e', lw=0.6, ls=':', alpha=0.6)
    ax.axhline(spd[-1], color=COLORS[mode], lw=0.6, ls='--', alpha=0.8)

    drift = r['speed_drift']
    ax.set_title(f'{mode.upper()}\ndrift: {drift:+.3f}', color=COLORS[mode], fontsize=10)
    ax.set_xlabel('Tick'); ax.grid(True)
    ax.legend(fontsize=7)

axes[0].set_ylabel('Mean speed')
fig.suptitle('Genetic evolution (speed) by environment', y=1.02)
plt.tight_layout(); plt.show()

### 3.1 Overlay — is the difference actually significant?

Same data, shared axes. Shows whether the 3 curves genuinely diverge or converge toward the same genetic attractor.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

for mode in EXPS:
    ts = data[mode][1]['timeseries']
    T  = range(len(ts['speed_mean']))
    ax.plot(T, ts['speed_mean'], color=COLORS[mode], lw=1.8, label=mode)
    ax.fill_between(T, ts['speed_min'], ts['speed_max'],
                    alpha=0.05, color=COLORS[mode])

ax.axhline(1.0, color='#8b949e', lw=0.7, ls='--', alpha=0.4)
ax.text(2, 1.01, 'neutral threshold', color='#8b949e', fontsize=8)
ax.set_xlabel('Tick'); ax.set_ylabel('Mean speed')
ax.set_title('Genetic convergence -- 3-mode comparison')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

## 4. Biodiversity — lineage survival

Each color = a family (lineage = founding ancestor ID). Area = number of agents from that family alive at each tick.

**What to look for**: do the same lineages survive across all 3 modes? Does a lineage that dominates in `flat` also dominate in `perlin`? If not, the terrain genuinely shifts selection pressure at the individual level.

In [ ]:
LINEAGE_COLORS = [
    '#ff6b6b','#ffd93d','#6bcb77','#4d96ff','#ff6bff',
    '#ff9f43','#48dbfb','#ff4757','#2ed573','#eccc68',
    '#a29bfe','#fd79a8','#55efc4','#fdcb6e','#74b9ff',
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for ax, mode in zip(axes, EXPS):
    lts  = data[mode][1]['lineage_ts']
    n    = len(next(iter(lts.values())))
    T    = np.arange(n)
    stack = np.zeros(n)

    for i in range(15):
        counts  = np.array(lts.get(str(i), [0]*n), dtype=float)
        new_top = stack + counts
        ax.fill_between(T, stack, new_top,
                        alpha=0.8,
                        color=LINEAGE_COLORS[i % len(LINEAGE_COLORS)],
                        label=f'L{i:02d}')
        stack = new_top

    r = data[mode][0]
    ax.set_title(f'{mode.upper()}\n{r["lineages_final"]}/{r["lineages_initial"]} lineages',
                 color=COLORS[mode], fontsize=10)
    ax.set_xlabel('Tick')
    ax.grid(True, axis='y', alpha=0.3)

axes[0].set_ylabel('Cumulative population')
fig.suptitle('Lineage survival by environment', y=1.02)
plt.tight_layout(); plt.show()

## 5. Final genetic distribution — who survived?

Scatter plot of surviving agents: **speed** vs **generation**. Reveals whether selection pressure produced distinct sub-populations, or whether everyone converged to the same speed.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True, sharex=True)

for ax, mode in zip(axes, EXPS):
    agents = data[mode][1]['final_agents']
    if not agents:
        ax.text(0.5, 0.5, 'EXTINCT', transform=ax.transAxes,
                ha='center', va='center', color='red', fontsize=14)
        continue

    speeds = [a['speed']      for a in agents]
    gens   = [a['generation'] for a in agents]
    colors = [LINEAGE_COLORS[a['lineage'] % len(LINEAGE_COLORS)] for a in agents]

    ax.scatter(speeds, gens, c=colors, s=25, alpha=0.7, linewidths=0)
    ax.axvline(np.mean(speeds), color=COLORS[mode], lw=1.2, ls='--',
               label=f'mean={np.mean(speeds):.2f}')

    r = data[mode][0]
    ax.set_title(f'{mode.upper()}\nσ={r["speed_std_final"]:.3f}',
                 color=COLORS[mode], fontsize=10)
    ax.set_xlabel('Speed'); ax.legend(fontsize=8); ax.grid(True)

axes[0].set_ylabel('Generation')
fig.suptitle('Final genetic distribution (color = lineage)', y=1.02)
plt.tight_layout(); plt.show()

## 6. Environmental pressure — resources over time

Mean resource per cell is a proxy for **survival pressure**. When it drops, the environment is more competitive. In Drought mode, it fluctuates as the terrain migrates.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))

for mode in EXPS:
    ts = data[mode][1]['timeseries']
    T  = range(len(ts['res_mean']))
    ax.plot(T, ts['res_mean'], color=COLORS[mode], lw=1.5, label=mode)

ax.set_xlabel('Tick'); ax.set_ylabel('Mean resource / cell')
ax.set_title('Environmental pressure over time')
ax.set_ylim(0, 5.5)
ax.axhline(5.0, color='#8b949e', lw=0.5, ls=':', alpha=0.4)
ax.text(2, 5.05, 'MAX_RES', color='#8b949e', fontsize=8)
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

## 7. Conclusion

### Observations

| Question | Observed answer |
|---|---|
| Does Perlin terrain slow convergence toward slow agents? | Yes, slightly. Speed drift is -0.697 in Perlin vs -0.721 in Flat. Fast agents have a small advantage near oases, which delays selection toward slow agents. |
| Do the same lineages dominate across all 3 modes? | No. Perlin is the only mode where 6 lineages survive (vs 5 in Flat and Drought). Spatial heterogeneity creates refuges that protect families that would be eliminated elsewhere. |
| Does Drought reverse selection pressure on speed? | Not really. It creates an early population and resource rebound (agents can't follow the moving oases, so resources accumulate), but at 2500 ticks it behaves like Flat. The drift is constant and predictable, so the population adapts and falls back to the same attractor. |
| Is genetic diversity (σ speed) higher in Perlin? | Yes, slightly: σ=0.117 in Perlin vs 0.104–0.105 elsewhere. Heterogeneous space protects slightly different speeds across zones. Not dramatic, but measurable. |

### Interpretation

The most striking result: all 3 modes converge to the same final speed (~0.63–0.66). The environment changes the *trajectory* but not the *destination*. Looking at the equations, this makes sense — the model has a fixed mathematical equilibrium determined by movement cost and food gain. No matter where resources are placed, the optimal speed stays the same.

What's interesting regardless: Perlin maintains slightly higher genetic diversity and protects one extra lineage. A subtle but real effect — space creates niches.

### What would it take to go further

- Add a **second genetic trait** coupled to speed (e.g. perception range) so the environment has a lasting effect on the genome, not just a transient one
- Test with an **unpredictable drought** (random direction) rather than a constant drift — fast agents might regain a durable advantage there
- Run multiple seeds and average results — current data is a single seed, hard to separate environmental effect from randomness
- Vary `MUTATION_STD`: does stronger mutation preserve diversity in Perlin?